# Named Entity Recognition

In [22]:
# Alternative implementation using NLTK since spaCy has installation issues
import nltk
import re
from collections import defaultdict

# Download required NLTK data - simplified approach
required_packages = [
    'punkt', 
    'punkt_tab', 
    'averaged_perceptron_tagger', 
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker', 
    'maxent_ne_chunker_tab',
    'words'
]

print("Setting up NLTK packages...")
for package in required_packages:
    try:
        nltk.download(package, quiet=True)
        print(f"✓ {package}")
    except Exception as e:
        print(f"⚠ {package}: {str(e)}")

print("\nNLTK setup complete!")

Setting up NLTK packages...
✓ punkt
✓ punkt_tab
✓ averaged_perceptron_tagger
✓ averaged_perceptron_tagger_eng
✓ maxent_ne_chunker
✓ maxent_ne_chunker_tab
✓ words

NLTK setup complete!
✓ maxent_ne_chunker_tab
✓ words

NLTK setup complete!


In [2]:
# Create a custom NER pipeline using NLTK
class NLTKNERPipeline:
    def __init__(self):
        self.labels = ['PERSON', 'ORGANIZATION', 'GPE', 'MONEY', 'DATE', 'TIME']
        # Pattern for money detection
        self.money_pattern = re.compile(r'\$\d+(?:,\d{3})*(?:\.\d{2})?(?:\s*(?:million|billion|thousand))?', re.IGNORECASE)
        
    def __call__(self, text):
        return self.process(text)
    
    def process(self, text):
        # Tokenize and tag
        tokens = nltk.word_tokenize(text)
        pos_tags = nltk.pos_tag(tokens)
        
        # Named Entity Recognition
        tree = nltk.ne_chunk(pos_tags, binary=False)
        
        entities = []
        current_entity = []
        current_label = None
        start_pos = 0
        
        # Extract entities from the tree
        for item in tree:
            if hasattr(item, 'label'):  # It's a named entity
                label = item.label()
                for word, tag in item:
                    current_entity.append(word)
                    if not current_label:
                        current_label = label
            else:  # It's a regular word
                if current_entity:
                    # Save the current entity
                    entity_text = ' '.join(current_entity)
                    start_char = text.find(entity_text, start_pos)
                    end_char = start_char + len(entity_text)
                    entities.append(Entity(entity_text, current_label, start_char, end_char))
                    start_pos = end_char
                    current_entity = []
                    current_label = None
        
        # Handle any remaining entity
        if current_entity:
            entity_text = ' '.join(current_entity)
            start_char = text.find(entity_text, start_pos)
            end_char = start_char + len(entity_text)
            entities.append(Entity(entity_text, current_label, start_char, end_char))
        
        # Add money entities using regex
        for match in self.money_pattern.finditer(text):
            entities.append(Entity(match.group(), 'MONEY', match.start(), match.end()))
        
        return NERResult(text, entities)

class Entity:
    def __init__(self, text, label_, start_char, end_char):
        self.text = text
        self.label_ = label_
        self.start_char = start_char
        self.end_char = end_char

class NERResult:
    def __init__(self, text, entities):
        self.text = text
        self.ents = entities

# Create the NER pipeline
ner_pipeline = NLTKNERPipeline()
print("NLTK NER pipeline created successfully!")

NLTK NER pipeline created successfully!


In [3]:
ner_pipeline_labels = ner_pipeline.labels
ner_pipeline_labels

['PERSON', 'ORGANIZATION', 'GPE', 'MONEY', 'DATE', 'TIME']

In [4]:
# Create explain function similar to spaCy
def explain(label):
    explanations = {
        'ORGANIZATION': 'Companies, agencies, institutions, etc.',
        'PERSON': 'People, including fictional.',
        'GPE': 'Countries, cities, states.',
        'MONEY': 'Monetary values, including unit.',
        'DATE': 'Absolute or relative dates or periods.',
        'TIME': 'Times smaller than a day.'
    }
    return explanations.get(label, f"Entity type: {label}")

explain("ORGANIZATION")

'Companies, agencies, institutions, etc.'

In [5]:
explain("MONEY")

'Monetary values, including unit.'

In [6]:
explain("GPE")

'Countries, cities, states.'

Sample text is a snippet from the [LinkedIn wikipedia page](https://en.wikipedia.org/wiki/LinkedIn)

In [8]:
sample_text = """ The company was founded in December 2002 by Reid Hoffman and the founding team members from PayPal and Socialnet.com (Allen Blue, Eric Ly, Jean-Luc Vaillant, Lee Hower, Konstantin Guericke, Stephen Beitzel, David Eves, Ian McNish, Yan Pujante, Chris Saccheri).In late 2003, Sequoia Capital led the Series A investment in the company.In August 2004, LinkedIn reached 1 million users.In March 2006, LinkedIn achieved its first month of profitability.In April 2007, LinkedIn reached 10 million users.In February 2008, LinkedIn launched a mobile version of the site.

In June 2008, Sequoia Capital, Greylock Partners, and other venture capital firms purchased a 5% stake in the company for $53 million, giving the company a post-money valuation of approximately $1 billion. In November 2009, LinkedIn opened its office in Mumbai and soon thereafter in Sydney, as it started its Asia-Pacific team expansion. In 2010 LinkedIn opened an International Headquarters in Dublin, Ireland,received a $20 million investment from Tiger Global Management LLC at a valuation of approximately $2 billion,announced its first acquisition, Mspoke,and improved its 1% premium subscription ratio. In October of that year, Silicon Valley Insider ranked the company No. 10 on its Top 100 List of most valuable startups. By December, the company was valued at $1.575 billion in private markets. LinkedIn started its India operations in 2009 and a major part of the first year was dedicated to understanding professionals in India and educating members to leverage LinkedIn for career development.

LinkedIn office building at 222 Second Street in San Francisco (opened in March 2016)

LinkedIn office in Toronto inside the Toronto Eaton Centre

LinkedIn filed for an initial public offering in January 2011. The company traded its first shares on May 19, 2011, under the NYSE symbol "LNKD", at $45 per share. Shares of LinkedIn rose as much as 171% on their first day of trade on the New York Stock Exchange and closed at $94.25, more than 109% above IPO price. Shortly after the IPO, the site's underlying infrastructure was revised to allow accelerated revision-release cycles.In 2011 LinkedIn earned $154.6 million in advertising revenue alone, surpassing Twitter, which earned $139.5 million.LinkedIn's fourth-quarter 2011, earnings soared because of the company's increase in success in the social media world.[33] By this point LinkedIn had about 2,100 full-time employees compared to the 500 that it had in 2010.

In April 2014 LinkedIn announced that it had leased 222 Second Street, a 26-story building under construction in San Francisco's SoMa district, to accommodate up to 2,500 of its employees, with the lease covering 10 years.The goal was to join all San Francisco-based staff (1,250 as of January 2016) in one building, bringing sales and marketing employees together with the research and development team.They started to move in in March 2016. In February 2016 following an earnings report, LinkedIn's shares dropped 43.6% within a single day, down to $108.38 per share. LinkedIn lost $10 billion of its market capitalization that day.

In 2016 access to LinkedIn was blocked by Russian authorities for non-compliance with the 2015 national legislation that requires social media networks to store citizens' personal data on servers located in Russia.

In June 2016 Microsoft announced that it would acquire LinkedIn for $196 a share, a total value of $26.2 billion and the second largest acquisition made by Microsoft to date. The acquisition would be an all-cash, debt-financed transaction. Microsoft would allow LinkedIn to "retain its distinct brand, culture and independence", with Weiner to remain as CEO, who would then report to Microsoft CEO Satya Nadella. Analysts believed Microsoft saw the opportunity to integrate LinkedIn with its Office product suite to help better integrate the professional network system with its products. The deal was completed on December 8, 2016.

In late 2016 LinkedIn announced a planned increase of 200 new positions in its Dublin office, which would bring the total employee count to 1,200.Since 2017 94% of B2B marketers use LinkedIn to distribute content.

Soon after LinkedIn's acquisition by Microsoft, LinkedIn's new desktop version was introduced.The new version was meant to make the user experience seamless across mobile and desktop. Some of the changes were made according to the feedback received from the previously launched mobile app. Features that were not heavily used were removed. For example, the contact tagging and filtering features are not supported anymore.

Following the launch of the new user interface (UI), some users, complained about the missing features which were there in the older version, slowness, and bugs in it. The issues were faced by both free and premium users, and with both the desktop version and the mobile version of the site.

In 2019 LinkedIn launched globally the feature Open for Business that enables freelancers to be discovered on the platform.LinkedIn Events was launched in the same year.

In June 2020 Jeff Weiner stepped down as CEO and become executive chairman after 11 years in the role. Ryan Roslansky stepped up as CEO from his previous position as the senior vice president of product.In late July 2020, LinkedIn announced it laid off 960 employees, about 6 percent of total workforce, from the talent acquisition and global sales teams. In an email to all employees, CEO Ryan Roslansky said the cuts were due to effects of the global COVID-19 pandemic.In April 2021 CyberNews claimed that 500 million LinkedIn's accounts have leaked online.However, LinkedIn stated that "We have investigated an alleged set of LinkedIn data that has been posted for sale and have determined that it is actually an aggregation of data from a number of websites and companies".

In June 2021 PrivacySharks claimed that more than 700 million LinkedIn records was on sale on a hacker forum.LinkedIn later stated that this is not a breach, but scraped data which is also a violation of their Terms of Service.

Microsoft ended LinkedIn operations in China in October 2021"""

In [9]:
len(sample_text.split('.'))

59

In [23]:
ner_text = ner_pipeline(sample_text)

In [24]:
for word in ner_text.ents:
    print(word.text,word.label_,word.start_char, word.end_char)

Reid Hoffman PERSON 45 57
PayPal ORGANIZATION 93 99
Allen Blue PERSON 119 129
Eric Ly PERSON 131 138
Jean-Luc Vaillant PERSON 140 157
Lee Hower PERSON 159 168
Konstantin Guericke PERSON 170 189
Stephen Beitzel PERSON 191 206
David Eves PERSON 208 218
Ian McNish PERSON 220 230
Yan Pujante ORGANIZATION 232 243
Chris Saccheri PERSON 245 259
Sequoia Capital PERSON 275 290
LinkedIn ORGANIZATION 350 358
LinkedIn ORGANIZATION 398 406
LinkedIn ORGANIZATION 464 472
LinkedIn ORGANIZATION 516 524
Sequoia Capital PERSON 579 594
Greylock Partners GPE 596 613
LinkedIn ORGANIZATION 789 797
Mumbai GPE 819 825
Sydney GPE 849 855
LinkedIn ORGANIZATION 912 920
International Headquarters ORGANIZATION 931 957
Dublin GPE 961 967
Ireland GPE 969 976
Tiger Global PERSON 1016 1028
Mspoke PERSON 1120 1126
Silicon Valley Insider PERSON 1200 1222
LinkedIn ORGANIZATION 1370 1378
India GPE 1391 1396
India GPE 1499 1504
LinkedIn ORGANIZATION 1539 1547
LinkedIn ORGANIZATION 1573 1581
San Francisco GPE 1622 1635
Linke

In [25]:
len([ent for ent in ner_text.ents if ent.label_ == 'MONEY'])

13

In [26]:
# Simple visualization of entities (alternative to displacy)
def display_entities(ner_result):
    text = ner_result.text
    entities = sorted(ner_result.ents, key=lambda x: x.start_char)
    
    print("Named Entity Recognition Results:")
    print("=" * 50)
    
    # Color mapping for different entity types
    colors = {
        'PERSON': '🟦',
        'ORGANIZATION': '🟩', 
        'GPE': '🟨',
        'MONEY': '🟪',
        'DATE': '🟧',
        'TIME': '🟫'
    }
    
    last_end = 0
    output = ""
    
    for entity in entities:
        # Add text before entity
        output += text[last_end:entity.start_char]
        
        # Add highlighted entity
        color = colors.get(entity.label_, '⬜')
        output += f"{color}[{entity.text}]({entity.label_})"
        
        last_end = entity.end_char
    
    # Add remaining text
    output += text[last_end:]
    
    print(output)
    print("\nLegend:")
    for label, symbol in colors.items():
        print(f"{symbol} {label}: {explain(label)}")

display_entities(ner_text)

Named Entity Recognition Results:
 The company was founded in December 2002 by 🟦[Reid Hoffman](PERSON) and the founding team members from 🟩[PayPal](ORGANIZATION) and Socialnet.com (🟦[Allen Blue](PERSON), 🟦[Eric Ly](PERSON), 🟦[Jean-Luc Vaillant](PERSON), 🟦[Lee Hower](PERSON), 🟦[Konstantin Guericke](PERSON), 🟦[Stephen Beitzel](PERSON), 🟦[David Eves](PERSON), 🟦[Ian McNish](PERSON), 🟩[Yan Pujante](ORGANIZATION), 🟦[Chris Saccheri](PERSON)).In late 2003, 🟦[Sequoia Capital](PERSON) led the Series A investment in the company.In August 2004, 🟩[LinkedIn](ORGANIZATION) reached 1 million users.In March 2006, 🟩[LinkedIn](ORGANIZATION) achieved its first month of profitability.In April 2007, 🟩[LinkedIn](ORGANIZATION) reached 10 million users.In February 2008, 🟩[LinkedIn](ORGANIZATION) launched a mobile version of the site.

In June 2008, 🟦[Sequoia Capital](PERSON), 🟨[Greylock Partners](GPE), and other venture capital firms purchased a 5% stake in the company for 🟪[$53 million](MONEY), giving the com